# Easy Agent — Model Communication Layer

> **Scope:** `src/services/api/`, `src/services/mcp/`, `src/services/skills/`, `src/types/message.ts`, `src/types/mcp.ts`, `src/types/types.ts`, `src/utils/streamDebug.ts`, `src/utils/tokens.ts`

The Model Communication Layer is the lowest layer in Easy Agent's five-layer architecture. It provides three foundational capabilities:

1. **Streaming API communication** with LLMs (Anthropic SDK)
2. **External tool server integration** via the Model Context Protocol (MCP)
3. **Reusable prompt template management** through the Skills system

Every token that flows between Easy Agent and an LLM passes through this layer.

---

## 1. Imports, Environment & Path Discovery

In [ ]:
import os, json, re, math
from pathlib import Path
from dataclasses import dataclass, field
from typing import Union, Optional, Callable, Awaitable, Any
from enum import Enum

# Path discovery — find project root from any subdirectory
_cwd = Path(".").resolve()
PROJECT_ROOT = _cwd
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "package.json").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise FileNotFoundError(f"Could not find package.json starting from {_cwd}")

# Load environment
from dotenv import load_dotenv
# .env lives at project root
_env_path = PROJECT_ROOT / ".env"
if not _env_path.exists():
    _env_path = PROJECT_ROOT.parent / ".env"
load_dotenv(_env_path, override=True)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"ANTHROPIC_AUTH_TOKEN set: {bool(os.environ.get('ANTHROPIC_AUTH_TOKEN'))}")

## 2. Type Foundations — Message & Content Block Types

The message type system maps directly to the Anthropic Messages API format. Every piece of content exchanged between Easy Agent and the LLM is expressed through these types.

**Source:** `src/types/message.ts`

In [ ]:
# ─── Content Block Types ───────────────────────────────────────────

@dataclass
class TextBlock:
    type: str = "text"
    text: str = ""

@dataclass
class ToolUseBlock:
    type: str = "tool_use"
    id: str = ""
    name: str = ""
    input: dict = field(default_factory=dict)

@dataclass
class ToolResultBlock:
    type: str = "tool_result"
    tool_use_id: str = ""
    content: Union[str, list] = ""
    is_error: bool = False

@dataclass
class ThinkingBlock:
    """Extended-thinking content block. The signature field is required by the
    API when echoing the message back on the next turn."""
    type: str = "thinking"
    thinking: str = ""
    signature: Optional[str] = None

ContentBlock = Union[TextBlock, ToolUseBlock, ToolResultBlock, ThinkingBlock]

# ─── Message Types ─────────────────────────────────────────────────

@dataclass
class UserMessage:
    role: str = "user"
    content: Union[str, list] = ""

@dataclass
class AssistantMessage:
    role: str = "assistant"
    content: Union[str, list] = ""

Message = Union[UserMessage, AssistantMessage]

# ─── Usage Tracking ────────────────────────────────────────────────

@dataclass
class Usage:
    input_tokens: int = 0
    output_tokens: int = 0
    cache_creation_input_tokens: Optional[int] = None
    cache_read_input_tokens: Optional[int] = None

print("Message types defined:")
print(f"  ContentBlock variants: TextBlock, ToolUseBlock, ToolResultBlock, ThinkingBlock")
print(f"  Message variants: UserMessage, AssistantMessage")

## 3. Stream Event Types

The streaming system yields a discriminated union of events that the agentic loop and UI consume incrementally.

| Event | When Emitted | Consumer |
|-------|-------------|----------|
| `message_start` | First event from the API | Stream debug logger |
| `text` | Each text delta from the model | UI (real-time rendering) |
| `tool_use_start` | New `tool_use` content block begins | UI (show tool card) |
| `tool_use_input` | Each JSON delta for tool arguments | UI (progressive display) |
| `message_done` | Stream complete | Agentic loop (assemble response) |
| `error` | Any error during streaming | Agentic loop (abort turn) |

**Source:** `src/types/message.ts` (lines 72–112)

In [ ]:
@dataclass
class StreamTextEvent:
    type: str = "text"
    text: str = ""

@dataclass
class StreamToolUseStartEvent:
    type: str = "tool_use_start"
    id: str = ""
    name: str = ""

@dataclass
class StreamToolUseInputEvent:
    type: str = "tool_use_input"
    id: str = ""
    partial_json: str = ""

@dataclass
class StreamMessageStartEvent:
    type: str = "message_start"
    messageId: str = ""

@dataclass
class StreamMessageDoneEvent:
    type: str = "message_done"
    stopReason: str = ""
    usage: Usage = field(default_factory=Usage)

@dataclass
class StreamErrorEvent:
    type: str = "error"
    error: Exception = field(default_factory=Exception)

StreamEvent = Union[
    StreamTextEvent, StreamToolUseStartEvent, StreamToolUseInputEvent,
    StreamMessageStartEvent, StreamMessageDoneEvent, StreamErrorEvent,
]

@dataclass
class StreamResult:
    assistantMessage: AssistantMessage = field(default_factory=AssistantMessage)
    usage: Usage = field(default_factory=Usage)
    stopReason: str = ""

print("Stream event types defined: 6 variants + StreamResult")

## 4. API Client Subsystem — Singleton & Configuration

The API client is a thin wrapper around the Anthropic SDK. It manages a lazily-initialized singleton client instance.

**Key constants:**
| Constant | Value | Purpose |
|----------|-------|---------|
| `CAPPED_DEFAULT_MAX_TOKENS` | 8,000 | Default output cap for normal turns |
| `ESCALATED_MAX_TOKENS` | 64,000 | Retry cap when output is truncated |
| `COMPACT_MAX_OUTPUT_TOKENS` | 20,000 | Output cap for compaction calls |
| `MAX_OUTPUT_TOKENS_RECOVERY_LIMIT` | 3 | Max continuation attempts |

**Source:** `src/services/api/client.ts`

In [ ]:
import anthropic

# ─── Default Configuration ─────────────────────────────────────────

DEFAULT_MODEL = os.environ.get("ANTHROPIC_MODEL", "claude-sonnet-4-20250514")
CAPPED_DEFAULT_MAX_TOKENS = 8_000
ESCALATED_MAX_TOKENS = 64_000
COMPACT_MAX_OUTPUT_TOKENS = 20_000
MAX_OUTPUT_TOKENS_RECOVERY_LIMIT = 3
DEFAULT_MAX_TOKENS = CAPPED_DEFAULT_MAX_TOKENS

# ─── Client Singleton ──────────────────────────────────────────────

_client_instance: Optional[anthropic.Anthropic] = None


def get_anthropic_client(
    api_key: Optional[str] = None,
    base_url: Optional[str] = None,
) -> anthropic.Anthropic:
    """Get or create the Anthropic client instance.
    The SDK reads ANTHROPIC_AUTH_TOKEN from the environment.
    Optionally pass api_key to override. When no options are passed and
    a cached instance exists, returns the cache (fast path)."""
    global _client_instance

    if _client_instance is not None and api_key is None and base_url is None:
        return _client_instance

    client = anthropic.Anthropic(
        api_key=api_key or os.environ.get("ANTHROPIC_AUTH_TOKEN"),
        base_url=base_url or os.environ.get("ANTHROPIC_BASE_URL"),
    )

    # Only cache the "default" instance (no overrides)
    if api_key is None and base_url is None:
        _client_instance = client

    return client


def verify_api_key(api_key: Optional[str] = None) -> bool:
    """Verify the API key is valid by making a lightweight request."""
    try:
        client = get_anthropic_client(api_key=api_key)
        client.messages.create(
            model=DEFAULT_MODEL,
            max_tokens=1,
            messages=[{"role": "user", "content": "hi"}],
        )
        return True
    except Exception:
        return False


def reset_client() -> None:
    """Reset the cached client instance."""
    global _client_instance
    _client_instance = None


print(f"DEFAULT_MODEL: {DEFAULT_MODEL}")
print(f"Client singleton ready (lazy init)")

## 5. Streaming Engine — The Core Communication Primitive

`streamMessage()` is the most critical function in the entire communication layer. It is an async generator that:
1. Opens an SSE connection to the Anthropic API
2. Yields incremental `StreamEvent` objects as they arrive
3. Accumulates the full response internally (per-block JSON buffers)
4. Returns a `StreamResult` with the assembled `AssistantMessage`

**Critical implementation detail:** Tool input JSON is accumulated **per content-block index**, not in a single shared buffer. This prevents data corruption when providers emit overlapping content blocks.

**Source:** `src/services/api/streaming.ts` (lines 63–291)

> *Simplified:* Stream debug logging and AbortSignal support removed for clarity.

In [ ]:
def stream_message(
    messages: list[dict],
    model: str = None,
    max_tokens: int = None,
    system: str = None,
    tools: list[dict] = None,
) -> tuple[list, "StreamResult"]:
    """Send a streaming request to the Anthropic API, collect events, return result.

    In the TypeScript source this is an AsyncGenerator. Python async generators
    cannot return a value, so we collect events into a list and return both
    the event list and the final StreamResult.

    NOTE: The TypeScript version yields events incrementally to the UI. Here we
    demonstrate the same accumulation logic but return everything at once.
    """
    client = get_anthropic_client()
    model = model or DEFAULT_MODEL
    max_tokens = max_tokens or DEFAULT_MAX_TOKENS

    # Build the API request
    request_params = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": messages,
    }
    if system:
        request_params["system"] = system
    if tools and len(tools) > 0:
        request_params["tools"] = tools

    # State accumulators — tool input JSON tracked PER content-block index
    content_blocks: list = []
    tool_input_json_by_index: dict[int, str] = {}
    message_id = ""
    stop_reason = ""
    usage = Usage()
    events_collected = []

    try:
        with client.messages.stream(**request_params) as stream:
            for event in stream:
                event_type = event.type

                # ── Message lifecycle ──────────────────────────────
                if event_type == "message_start":
                    message_id = event.message.id
                    if event.message.usage:
                        usage.input_tokens = event.message.usage.input_tokens
                        usage.output_tokens = event.message.usage.output_tokens
                        if hasattr(event.message.usage, "cache_creation_input_tokens"):
                            usage.cache_creation_input_tokens = getattr(
                                event.message.usage, "cache_creation_input_tokens", None
                            )
                        if hasattr(event.message.usage, "cache_read_input_tokens"):
                            usage.cache_read_input_tokens = getattr(
                                event.message.usage, "cache_read_input_tokens", None
                            )
                    events_collected.append(StreamMessageStartEvent(messageId=message_id))

                elif event_type == "message_delta":
                    if hasattr(event, "usage") and event.usage:
                        usage.output_tokens = event.usage.output_tokens
                    stop_reason = getattr(event.delta, "stop_reason", "") or ""

                elif event_type == "message_stop":
                    events_collected.append(StreamMessageDoneEvent(stopReason=stop_reason, usage=usage))

                # ── Content block lifecycle ────────────────────────
                elif event_type == "content_block_start":
                    index = event.index
                    # Extend list to fit index
                    while len(content_blocks) <= index:
                        content_blocks.append(None)

                    if event.content_block.type == "text":
                        content_blocks[index] = TextBlock(text="")

                    elif event.content_block.type == "thinking":
                        content_blocks[index] = ThinkingBlock(
                            thinking=getattr(event.content_block, "thinking", "") or ""
                        )

                    elif event.content_block.type == "tool_use":
                        block = event.content_block
                        seed_input = (
                            block.input
                            if hasattr(block, "input") and isinstance(block.input, dict)
                            else {}
                        )
                        content_blocks[index] = ToolUseBlock(
                            id=block.id, name=block.name, input=seed_input
                        )
                        tool_input_json_by_index[index] = ""
                        events_collected.append(StreamToolUseStartEvent(id=block.id, name=block.name))

                elif event_type == "content_block_delta":
                    delta = event.delta
                    index = event.index
                    delta_type = delta.type

                    if delta_type == "text_delta":
                        content_blocks[index].text += delta.text
                        events_collected.append(StreamTextEvent(text=delta.text))

                    elif delta_type == "thinking_delta":
                        block = content_blocks[index]
                        if block and block.type == "thinking":
                            block.thinking += getattr(delta, "thinking", "") or ""

                    elif delta_type == "signature_delta":
                        block = content_blocks[index]
                        if block and block.type == "thinking":
                            sig = getattr(delta, "signature", "") or ""
                            block.signature = (block.signature or "") + sig

                    elif delta_type == "input_json_delta":
                        prev = tool_input_json_by_index.get(index, "")
                        tool_input_json_by_index[index] = prev + delta.partial_json
                        id_block = content_blocks[index]
                        if id_block and id_block.type == "tool_use":
                            events_collected.append(StreamToolUseInputEvent(
                                id=id_block.id, partial_json=delta.partial_json
                            ))

                elif event_type == "content_block_stop":
                    index = event.index
                    block = content_blocks[index]
                    accumulated = tool_input_json_by_index.get(index)
                    if block and block.type == "tool_use" and accumulated:
                        try:
                            block.input = json.loads(accumulated)
                        except json.JSONDecodeError:
                            block.input = {"_raw": accumulated}
                    tool_input_json_by_index.pop(index, None)

    except Exception as error:
        events_collected.append(StreamErrorEvent(error=error))

    result = StreamResult(
        assistantMessage=AssistantMessage(
            role="assistant",
            content=[b for b in content_blocks if b],
        ),
        usage=usage,
        stopReason=stop_reason,
    )
    return events_collected, result


print("stream_message() defined — core streaming primitive")

### 5.1 Demo: Streaming a Simple Message

Run the streaming generator against the real Anthropic API and collect events.

In [ ]:
try:
    events, result = stream_message(
        messages=[{"role": "user", "content": "What is 2+2? Reply in one word."}],
        max_tokens=50,
    )

    # Reconstruct text from text events (as the UI would do incrementally)
    final_text = "".join(e.text for e in events if e.type == "text")

    print(f"Events received: {len(events)}")
    print(f"Event types: {[e.type for e in events]}")
    print(f"Final text: {final_text!r}")
    print(f"Stop reason: {result.stopReason}")
    print(f"Usage: input={result.usage.input_tokens}, output={result.usage.output_tokens}")
except Exception as e:
    print(f"API call failed (likely rate limit): {type(e).__name__}: {str(e)[:120]}")
    print("This is expected if running without API credits or hitting rate limits.")

### 5.2 Non-Streaming `createMessage()` and Escalated Retry

For internal tasks (compaction, classification) where incremental output is unnecessary, `createMessage()` provides a simpler synchronous call.

The escalation strategy: first try at 8K tokens, if truncated (stop_reason == "max_tokens") retry at 64K.

**Source:** `src/services/api/streaming.ts` (lines 300–400)

In [ ]:
def create_message(
    messages: list[dict],
    model: str = None,
    max_tokens: int = None,
    system: str = None,
    tools: list[dict] = None,
) -> dict:
    """Simple non-streaming call for quick one-off requests."""
    client = get_anthropic_client()
    model = model or DEFAULT_MODEL
    max_tokens = max_tokens or DEFAULT_MAX_TOKENS

    kwargs = {"model": model, "max_tokens": max_tokens, "messages": messages}
    if system:
        kwargs["system"] = system
    if tools and len(tools) > 0:
        kwargs["tools"] = tools

    response = client.messages.create(**kwargs)

    content_blocks = []
    for block in response.content:
        if block.type == "text":
            content_blocks.append(TextBlock(text=block.text))
        elif block.type == "tool_use":
            content_blocks.append(ToolUseBlock(
                id=block.id, name=block.name, input=block.input
            ))

    usage_result = Usage(
        input_tokens=response.usage.input_tokens,
        output_tokens=response.usage.output_tokens,
    )

    return {
        "content": content_blocks,
        "usage": usage_result,
        "stopReason": response.stop_reason or "end_turn",
    }


# Demo
try:
    result = create_message(
        messages=[{"role": "user", "content": "Say 'hello' and nothing else."}],
        max_tokens=20,
    )
    print(f"stopReason: {result['stopReason']}")
    print(f"content: {result['content'][0].text!r}")
    print(f"usage: input={result['usage'].input_tokens}, output={result['usage'].output_tokens}")
except Exception as e:
    print(f"API call failed: {type(e).__name__}: {str(e)[:120]}")
    print("This is expected if running without API credits or hitting rate limits.")

## 6. Token Estimation and Budget Management

The system uses character-based heuristics (no native tokenizer dependency):

| Content Type | Chars/Token | Rationale |
|-------------|-------------|-----------|
| Plain text | 4 | Average English text ratio |
| JSON (tool input/output) | 2 | JSON is more token-dense |
| Binary (images) | Fixed 2,000 | Conservative estimate |
| Message overhead | 12 tokens | Role + structural framing |
| Tool block overhead | 24 tokens | Name + schema metadata |

A `4/3` inflation factor accounts for systematic underestimation.

**Source:** `src/utils/tokens.ts`

In [ ]:
# ─── Token Constants ───────────────────────────────────────────────

MODEL_CONTEXT_WINDOW_DEFAULT = 200_000
MAX_OUTPUT_TOKENS_FOR_SUMMARY = 20_000
AUTOCOMPACT_BUFFER_TOKENS = 13_000
WARNING_THRESHOLD_BUFFER_TOKENS = 20_000
MANUAL_COMPACT_BUFFER_TOKENS = 3_000

TEXT_CHARS_PER_TOKEN = 4
JSON_CHARS_PER_TOKEN = 2
MESSAGE_OVERHEAD_TOKENS = 12
TOOL_BLOCK_OVERHEAD_TOKENS = 24
FIXED_BINARY_BLOCK_TOKENS = 2_000

MODEL_CONTEXT_WINDOWS = {
    "claude-opus-4-20250514": 200_000,
    "claude-sonnet-4-20250514": 200_000,
    "claude-haiku-3-20250307": 200_000,
    "claude-3-5-sonnet-20241022": 200_000,
    "claude-3-5-haiku-20241022": 200_000,
    "claude-3-opus-20240229": 200_000,
}


def get_context_window_for_model(model: str) -> int:
    env_override = os.environ.get("CLAUDE_CODE_MAX_CONTEXT_TOKENS")
    if env_override:
        parsed = int(env_override)
        if parsed > 0:
            return parsed
    if model in MODEL_CONTEXT_WINDOWS:
        return MODEL_CONTEXT_WINDOWS[model]
    # Fuzzy match
    for key, value in MODEL_CONTEXT_WINDOWS.items():
        if key in model or model in key:
            return value
    return MODEL_CONTEXT_WINDOW_DEFAULT


def get_effective_context_window(model: str) -> int:
    """Context window minus reserved space for output."""
    window = get_context_window_for_model(model)
    reserved = min(MAX_OUTPUT_TOKENS_FOR_SUMMARY, window // 5)
    return window - reserved


def rough_token_estimation(content: str, chars_per_token: int = TEXT_CHARS_PER_TOKEN) -> int:
    return max(1, round(len(content) / chars_per_token))


def estimate_content_block_tokens(content) -> int:
    """Estimate tokens for message content (string or block list)."""
    if isinstance(content, str):
        return rough_token_estimation(content)
    if not isinstance(content, list):
        return 0

    total = 0
    for block in content:
        if isinstance(block, dict):
            block_type = block.get("type", "")
        else:
            block_type = getattr(block, "type", "")

        if block_type == "text":
            text = block.get("text", "") if isinstance(block, dict) else block.text
            total += rough_token_estimation(text)
        elif block_type == "tool_use":
            total += TOOL_BLOCK_OVERHEAD_TOKENS
            name = block.get("name", "") if isinstance(block, dict) else block.name
            inp = block.get("input", {}) if isinstance(block, dict) else block.input
            total += rough_token_estimation(name)
            total += max(1, round(len(json.dumps(inp)) / JSON_CHARS_PER_TOKEN))
        elif block_type == "tool_result":
            c = block.get("content", "") if isinstance(block, dict) else block.content
            serialized = c if isinstance(c, str) else json.dumps(c)
            total += TOOL_BLOCK_OVERHEAD_TOKENS + max(1, round(len(serialized) / JSON_CHARS_PER_TOKEN))
        elif block_type in ("image", "document"):
            total += FIXED_BINARY_BLOCK_TOKENS
        else:
            total += max(1, round(len(json.dumps(block if isinstance(block, dict) else {})) / JSON_CHARS_PER_TOKEN))
    return total


def estimate_message_tokens(message: dict) -> int:
    return MESSAGE_OVERHEAD_TOKENS + estimate_content_block_tokens(message.get("content", ""))


def rough_token_count_for_messages(messages: list[dict]) -> int:
    """Estimate total tokens with 4/3 inflation factor."""
    raw = sum(estimate_message_tokens(m) for m in messages)
    return math.ceil((raw * 4) / 3)


print(f"Context window for {DEFAULT_MODEL}: {get_context_window_for_model(DEFAULT_MODEL):,}")
print(f"Effective context window: {get_effective_context_window(DEFAULT_MODEL):,}")

### 6.1 Context Window Budget Snapshot

The budget system computes thresholds for auto-compaction:
- **Effective context window** = total window - reserved for output (min(20K, 20%))
- **Auto-compact threshold** = effective - 13,000 buffer
- **Manual compact threshold** = effective - 3,000 buffer

In [ ]:
def get_token_count_from_usage(usage: Usage) -> int:
    return (
        usage.input_tokens
        + (usage.cache_creation_input_tokens or 0)
        + (usage.cache_read_input_tokens or 0)
        + usage.output_tokens
    )


def estimate_system_prompt_tokens(system_prompt: str) -> int:
    return rough_token_estimation(system_prompt) + MESSAGE_OVERHEAD_TOKENS


def token_count_with_estimation(
    messages: list[dict],
    usage: Optional[Usage] = None,
    usage_anchor_index: Optional[int] = None,
    system_prompt: Optional[str] = None,
) -> int:
    """Hybrid: use API usage as anchor when available, estimate only the suffix."""
    system_tokens = estimate_system_prompt_tokens(system_prompt) if system_prompt else 0

    if usage and usage_anchor_index is not None and usage_anchor_index >= 0:
        suffix = messages[usage_anchor_index + 1:]
        return get_token_count_from_usage(usage) + rough_token_count_for_messages(suffix) + system_tokens

    return rough_token_count_for_messages(messages) + system_tokens


def scale_buffer(buffer: int, effective_window: int) -> int:
    reference_window = 180_000
    if effective_window >= reference_window:
        return buffer
    return round(buffer * (effective_window / reference_window))


def build_token_budget_snapshot(
    messages: list[dict],
    model: str = None,
    usage: Optional[Usage] = None,
    usage_anchor_index: Optional[int] = None,
    system_prompt: Optional[str] = None,
) -> dict:
    """Build a complete budget snapshot for a conversation."""
    model = model or DEFAULT_MODEL
    estimated = token_count_with_estimation(messages, usage, usage_anchor_index, system_prompt)
    context_window = get_context_window_for_model(model)
    effective = get_effective_context_window(model)

    return {
        "estimated_conversation_tokens": estimated,
        "context_window": context_window,
        "effective_context_window": effective,
        "auto_compact_threshold": max(0, effective - scale_buffer(AUTOCOMPACT_BUFFER_TOKENS, effective)),
        "manual_compact_threshold": max(0, effective - scale_buffer(MANUAL_COMPACT_BUFFER_TOKENS, effective)),
    }


# Demo
sample_messages = [
    {"role": "user", "content": "Explain how transformers work in machine learning."},
    {"role": "assistant", "content": "Transformers use self-attention mechanisms..." * 50},
]
snapshot = build_token_budget_snapshot(sample_messages)
print("Token Budget Snapshot:")
for k, v in snapshot.items():
    print(f"  {k}: {v:,}")

## 7. MCP Subsystem — Type Definitions

The MCP type system models the full lifecycle of an MCP server connection: configuration → pending → connected/failed.

Three transport types are supported:
- **stdio** — local subprocess (default)
- **http** — Streamable HTTP (recommended for remote)
- **sse** — legacy Server-Sent Events

**Source:** `src/types/mcp.ts`

In [ ]:
# ─── MCP Configuration Types ───────────────────────────────────────

@dataclass
class McpStdioServerConfig:
    type: str = "stdio"
    command: str = ""
    args: list = field(default_factory=list)
    env: dict = field(default_factory=dict)

@dataclass
class McpHTTPServerConfig:
    type: str = "http"
    url: str = ""
    headers: dict = field(default_factory=dict)

@dataclass
class McpSSEServerConfig:
    type: str = "sse"
    url: str = ""
    headers: dict = field(default_factory=dict)

McpServerConfig = Union[McpStdioServerConfig, McpHTTPServerConfig, McpSSEServerConfig]

@dataclass
class ScopedMcpServerConfig:
    """Server config tagged with its origin scope (project overrides user)."""
    config: McpServerConfig = field(default_factory=McpStdioServerConfig)
    scope: str = "user"  # "user" or "project"

# ─── MCP Connection State Types ────────────────────────────────────

@dataclass
class ConnectedMcpServer:
    name: str = ""
    type: str = "connected"
    client: Any = None
    capabilities: Any = None
    server_info: Optional[dict] = None
    config: Any = None
    cleanup: Optional[Callable] = None

@dataclass
class FailedMcpServer:
    name: str = ""
    type: str = "failed"
    config: Any = None
    error: str = ""

@dataclass
class PendingMcpServer:
    name: str = ""
    type: str = "pending"
    config: Any = None
    started_at: int = 0

@dataclass
class DisabledMcpServer:
    name: str = ""
    type: str = "disabled"
    config: Any = None

McpServerConnection = Union[ConnectedMcpServer, FailedMcpServer, PendingMcpServer, DisabledMcpServer]

print("MCP types defined:")
print("  Configs: McpStdioServerConfig, McpHTTPServerConfig, McpSSEServerConfig")
print("  States: Connected, Failed, Pending, Disabled")

## 8. MCP Name Normalization & Tool Name Convention

The Anthropic API requires tool names to match `^[a-zA-Z0-9_-]{1,64}$`. MCP names allow broader characters, so invalid characters are replaced with `_`.

Tool name format: `mcp__<normalizedServer>__<normalizedTool>`

**Source:** `src/services/mcp/normalization.ts`, `src/services/mcp/mcpStringUtils.ts`

In [ ]:
def normalize_name_for_mcp(name: str) -> str:
    """Replace any non-alphanumeric/non-dash/non-underscore character with _."""
    return re.sub(r"[^a-zA-Z0-9_-]", "_", name)


def build_mcp_tool_name(server_name: str, tool_name: str) -> str:
    """Build the fully qualified MCP tool name."""
    return f"mcp__{normalize_name_for_mcp(server_name)}__{normalize_name_for_mcp(tool_name)}"


def is_mcp_tool_name(name: str) -> bool:
    """Cheap predicate: does this look like an MCP-prefixed tool name?"""
    return name.startswith("mcp__")


def parse_mcp_tool_name(full_name: str) -> Optional[dict]:
    """Parse an MCP tool name back into server/tool components."""
    parts = full_name.split("__")
    if len(parts) < 3 or parts[0] != "mcp" or not parts[1]:
        return None
    return {
        "server_name": parts[1],
        "tool_name": "__".join(parts[2:]),  # Rejoin if tool name contained __
    }


# Demo
examples = [
    ("my.server", "run_query"),
    ("github-mcp", "search_code"),
    ("Open AI Server", "chat.completions"),
]
print("Name normalization examples:")
for server, tool in examples:
    full = build_mcp_tool_name(server, tool)
    parsed = parse_mcp_tool_name(full)
    print(f"  ({server!r}, {tool!r}) -> {full!r}")
    print(f"    parse back: {parsed}")

## 9. MCP Configuration Loading & Validation

Configuration is loaded from two JSON files with project-overrides-user precedence:
- **User:** `~/.easy-agent/settings.json`
- **Project:** `<cwd>/.easy-agent/settings.json`

Each server config is validated per transport type. Invalid entries are dropped with a warning (never throws).

**Source:** `src/services/mcp/config.ts`

In [ ]:
def validate_stdio_config(name: str, obj: dict, scope: str) -> tuple[bool, Any]:
    """Validate a stdio server config. Returns (ok, value_or_error)."""
    if not isinstance(obj.get("command"), str) or not obj["command"].strip():
        return (False, f"mcpServers.{name} ({scope}): 'command' is required")

    if "args" in obj and not isinstance(obj["args"], list):
        return (False, f"mcpServers.{name} ({scope}): 'args' must be an array")

    if "args" in obj and any(not isinstance(a, str) for a in obj["args"]):
        return (False, f"mcpServers.{name} ({scope}): 'args' must contain only strings")

    if "env" in obj:
        if not isinstance(obj["env"], dict):
            return (False, f"mcpServers.{name} ({scope}): 'env' must be a string->string map")
        for k, v in obj["env"].items():
            if not isinstance(v, str):
                return (False, f"mcpServers.{name} ({scope}): env.{k} must be a string")

    return (True, McpStdioServerConfig(
        type="stdio",
        command=obj["command"],
        args=obj.get("args", []),
        env=obj.get("env", {}),
    ))


def validate_remote_config(name: str, obj: dict, scope: str, transport: str) -> tuple[bool, Any]:
    """Validate an http or sse server config."""
    if not isinstance(obj.get("url"), str) or not obj["url"].strip():
        return (False, f"mcpServers.{name} ({scope}): '{transport}' requires 'url'")

    from urllib.parse import urlparse
    try:
        result = urlparse(obj["url"])
        if not result.scheme:
            raise ValueError("no scheme")
    except Exception:
        return (False, f"mcpServers.{name} ({scope}): 'url' is not valid: {obj['url']}")

    if "headers" in obj:
        if not isinstance(obj["headers"], dict):
            return (False, f"mcpServers.{name} ({scope}): 'headers' must be a string->string map")

    config_cls = McpHTTPServerConfig if transport == "http" else McpSSEServerConfig
    return (True, config_cls(type=transport, url=obj["url"], headers=obj.get("headers", {})))


def validate_server_config(name: str, raw: Any, scope: str) -> tuple[bool, Any]:
    """Validate a single MCP server config entry."""
    if not raw or not isinstance(raw, dict):
        return (False, f"mcpServers.{name} must be an object")

    transport = raw.get("type")
    if transport is not None and transport not in ("stdio", "http", "sse"):
        return (False, f"mcpServers.{name} ({scope}): unsupported transport '{transport}'")

    if transport in ("http", "sse"):
        return validate_remote_config(name, raw, scope, transport)
    return validate_stdio_config(name, raw, scope)


def load_mcp_configs_from_dict(raw_settings: dict, scope: str) -> tuple[dict, list]:
    """Extract and validate MCP servers from a settings dict."""
    errors = []
    servers = {}

    mcp_servers = raw_settings.get("mcpServers", {})
    if not isinstance(mcp_servers, dict):
        errors.append(f"'mcpServers' must be an object")
        return servers, errors

    for name, raw_config in mcp_servers.items():
        ok, result = validate_server_config(name, raw_config, scope)
        if not ok:
            errors.append(result)
        else:
            servers[name] = {"config": result, "scope": scope}

    return servers, errors


# Demo: validate example configs
test_configs = {
    "mcpServers": {
        "filesystem": {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem"]},
        "github": {"type": "http", "url": "https://mcp.github.com/api"},
        "broken": {"type": "stdio"},  # missing command
        "invalid-type": {"type": "websocket", "url": "ws://localhost"},
    }
}

servers, errors = load_mcp_configs_from_dict(test_configs, "project")
print("Valid servers:")
for name, entry in servers.items():
    print(f"  {name}: {entry['config'].type} ({entry['scope']})")
print(f"\nValidation errors ({len(errors)}):")
for err in errors:
    print(f"  {err}")

## 10. MCP Registry — In-Memory Connection Store

The registry is a simple `Map<name, {connection, tools}>` that provides the runtime state for `/mcp` status display and tool routing.

**Source:** `src/services/mcp/registry.ts`

In [ ]:
class McpRegistry:
    """In-memory registry of MCP server connections + their tools."""

    def __init__(self):
        self._entries: dict[str, dict] = {}

    def set_entry(self, name: str, connection: McpServerConnection, tools: list) -> None:
        self._entries[name] = {"connection": connection, "tools": tools}

    def delete_entry(self, name: str) -> None:
        self._entries.pop(name, None)

    def get_entry(self, name: str) -> Optional[dict]:
        return self._entries.get(name)

    def get_all(self) -> list[dict]:
        return list(self._entries.values())

    def get_all_tools(self) -> list:
        return [tool for entry in self._entries.values() for tool in entry["tools"]]

    def clear(self) -> None:
        self._entries.clear()

    def __repr__(self):
        statuses = {name: e["connection"].type for name, e in self._entries.items()}
        return f"McpRegistry({statuses})"


# Demo
registry = McpRegistry()
registry.set_entry("filesystem", PendingMcpServer(name="filesystem", started_at=1000), [])
registry.set_entry("github", ConnectedMcpServer(name="github"), ["tool_a", "tool_b"])
print(registry)
print(f"Total tools: {len(registry.get_all_tools())}")

## 11. MCP Bootstrap — Orchestrated Startup Sequence

The bootstrap sequence:
1. Load + validate `mcpServers` from settings.json (user + project)
2. Register process cleanup (SIGINT/SIGTERM handler)
3. Seed "pending" placeholders for immediate UI feedback
4. Connect each server in parallel (`Promise.allSettled` / `asyncio.gather`)
5. For each connected server, fetch `tools/list`
6. Register tools into the global registry

**Key design:** Slow servers don't block fast ones — tools become available incrementally.

**Source:** `src/services/mcp/bootstrap.ts`

> *Simplified:* Process cleanup, actual SDK connections, and parallel I/O removed. Shows the orchestration logic.

In [ ]:
import time
import random

def bootstrap_mcp_simulated(config: dict) -> dict:
    """Simulated MCP bootstrap demonstrating the orchestration flow."""
    registry = McpRegistry()
    registry.clear()

    # Step 1: Load configs
    servers, config_errors = load_mcp_configs_from_dict(config, "project")
    print(f"Step 1: Loaded {len(servers)} server configs, {len(config_errors)} errors")

    # Step 2: Seed pending placeholders (immediate UI feedback)
    started_at = int(time.time() * 1000)
    for name, entry in servers.items():
        placeholder = PendingMcpServer(name=name, config=entry["config"], started_at=started_at)
        registry.set_entry(name, placeholder, [])
    print(f"Step 2: Seeded {len(servers)} pending placeholders")
    print(f"  Registry state: {registry}")

    # Step 3: Connect each server (simulated — would be parallel in real code)
    random.seed(42)  # deterministic for notebook
    for name, entry in servers.items():
        # Simulate: 80% chance of success
        if random.random() < 0.8:
            connected = ConnectedMcpServer(
                name=name,
                config=entry["config"],
                capabilities={"tools": True},
            )
            # Simulated tools
            mock_tools = [f"{name}_tool_{i}" for i in range(random.randint(1, 5))]
            registry.set_entry(name, connected, mock_tools)
            print(f"  [{name}] connected, {len(mock_tools)} tools")
        else:
            failed = FailedMcpServer(name=name, config=entry["config"], error="Connection timed out")
            registry.set_entry(name, failed, [])
            print(f"  [{name}] FAILED: Connection timed out")

    # Step 4: Summary
    all_tools = registry.get_all_tools()
    print(f"\nBootstrap complete:")
    print(f"  Total tools available: {len(all_tools)}")
    print(f"  Final registry: {registry}")
    return {"registry": registry, "tool_count": len(all_tools), "errors": config_errors}


# Run simulated bootstrap
test_settings = {
    "mcpServers": {
        "filesystem": {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem"]},
        "github": {"type": "http", "url": "https://mcp.github.com/api"},
        "memory": {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-memory"]},
    }
}

result = bootstrap_mcp_simulated(test_settings)

## 12. MCP Tool Discovery & Adapter Construction

When a server connects, `fetchToolsForConnection()` calls `tools/list` and wraps each MCP tool in a local `Tool` adapter. The adapter:
- Uses the **prefixed name** (`mcp__server__tool`) for the local registry
- Sends the **original name** to the MCP server on invocation
- Truncates descriptions at 2048 chars
- Maps `readOnlyHint` annotation to `isReadOnly()`
- Stringifies multi-block responses (text, image, resource) into a single string

**Source:** `src/services/mcp/fetchTools.ts`

In [ ]:
MAX_MCP_DESCRIPTION_LENGTH = 2048


def truncate_description(desc: Optional[str]) -> str:
    if not desc:
        return ""
    if len(desc) <= MAX_MCP_DESCRIPTION_LENGTH:
        return desc
    return desc[:MAX_MCP_DESCRIPTION_LENGTH] + "... [truncated]"


def stringify_mcp_content(content: list) -> str:
    """Map MCP CallToolResult.content[] blocks to a single string."""
    if not isinstance(content, list):
        return ""
    parts = []
    for block in content:
        block_type = block.get("type", "")
        if block_type == "text":
            parts.append(block.get("text", ""))
        elif block_type == "image":
            mime = block.get("mimeType", "?")
            data_len = len(block.get("data", ""))
            parts.append(f"[image: {mime}, {data_len} base64 chars]")
        elif block_type == "resource":
            r = block.get("resource", {})
            parts.append(r.get("text", f"[resource: {r.get('uri', '<no uri>')}]"))
        else:
            parts.append(f"[{block_type or 'unknown'} block]")
    return "\n".join(parts)


@dataclass
class McpToolAdapter:
    """Local Tool wrapping an MCP tool descriptor."""
    name: str = ""
    description: str = ""
    input_schema: dict = field(default_factory=dict)
    _is_read_only: bool = False
    _original_name: str = ""
    _server_name: str = ""

    def is_read_only(self) -> bool:
        return self._is_read_only

    def is_enabled(self) -> bool:
        return True

    def call(self, raw_input: dict) -> dict:
        """In real code, this forwards to connection.client.request({method: 'tools/call'})."""
        return {
            "method": "tools/call",
            "params": {"name": self._original_name, "arguments": raw_input},
        }


def build_tool_adapter(server_name: str, mcp_tool: dict) -> McpToolAdapter:
    """Build a local Tool from a single MCP tool descriptor."""
    full_name = build_mcp_tool_name(server_name, mcp_tool["name"])
    description = truncate_description(mcp_tool.get("description"))
    annotations = mcp_tool.get("annotations", {}) or {}
    is_read_only = annotations.get("readOnlyHint", False)
    input_schema = mcp_tool.get("inputSchema", {"type": "object", "properties": {}})

    return McpToolAdapter(
        name=full_name,
        description=description,
        input_schema=input_schema,
        _is_read_only=is_read_only,
        _original_name=mcp_tool["name"],
        _server_name=server_name,
    )


# Demo
sample_mcp_tools = [
    {
        "name": "read_file",
        "description": "Read a file from the local filesystem",
        "inputSchema": {"type": "object", "properties": {"path": {"type": "string"}}},
        "annotations": {"readOnlyHint": True},
    },
    {
        "name": "write_file",
        "description": "Write content to a file",
        "inputSchema": {"type": "object", "properties": {"path": {"type": "string"}, "content": {"type": "string"}}},
    },
]

print("Tool adapter construction:")
for tool_desc in sample_mcp_tools:
    adapter = build_tool_adapter("filesystem", tool_desc)
    print(f"  {adapter.name}")
    print(f"    description: {adapter.description}")
    print(f"    readOnly: {adapter.is_read_only()}")
    print(f"    call example: {adapter.call({'path': '/tmp/test.txt'})}\n")

## 13. MCP Client — Connection & Signal Escalation

The connection flow:
1. Check connection cache (keyed by name + full config — any config edit yields fresh cache)
2. Create transport bundle (stdio/http/sse)
3. MCP SDK Client handshake with 30s timeout (`Promise.race`)
4. On success: read capabilities, build cleanup function
5. On failure: return `FailedMcpServer` with stderr tail

**Stdio cleanup escalation** (prevents zombie processes):
```
SIGINT (100ms wait) → SIGTERM (400ms wait) → SIGKILL
```
Total cap ~500ms so CLI exit isn't held up.

**Source:** `src/services/mcp/client.ts` (lines 91–127 for escalation, 273–367 for doConnect)

In [ ]:
import signal as signal_module

CONNECT_TIMEOUT_MS = 30_000


def get_cache_key(name: str, config) -> str:
    """Cache key includes full transport config so config edits bust the cache."""
    if hasattr(config, "url"):  # http or sse
        return f"{name}:{json.dumps({'type': config.type, 'url': config.url, 'headers': getattr(config, 'headers', {})}, sort_keys=True)}"
    return f"{name}:{json.dumps({'type': 'stdio', 'command': config.command, 'args': config.args, 'env': config.env}, sort_keys=True)}"


def escalated_kill_demo(name: str, pid: Optional[int]) -> list[str]:
    """Demonstrate the signal escalation strategy (does NOT actually kill)."""
    if not pid:
        return ["no pid"]
    steps = []

    # Step 1: SIGINT
    steps.append(f"kill({pid}, SIGINT)")
    steps.append("sleep(100ms)")
    steps.append("alive_check(pid)")

    # Step 2: SIGTERM (if still alive)
    steps.append(f"kill({pid}, SIGTERM)")
    steps.append("sleep(400ms)")
    steps.append("alive_check(pid)")

    # Step 3: SIGKILL (if still alive)
    steps.append(f"kill({pid}, SIGKILL)")
    return steps


# Demo: cache key generation
configs = [
    ("fs", McpStdioServerConfig(command="npx", args=["-y", "server-fs"])),
    ("fs", McpStdioServerConfig(command="npx", args=["-y", "server-fs", "--root=/tmp"])),
    ("api", McpHTTPServerConfig(type="http", url="https://api.example.com")),
]
print("Cache keys (any config change -> new key):")
for name, cfg in configs:
    print(f"  {get_cache_key(name, cfg)}")

print(f"\nSignal escalation steps for pid=12345:")
for step in escalated_kill_demo("test-server", 12345):
    print(f"  {step}")

## 14. Skills Subsystem — Type Definitions

A Skill is a Markdown file with YAML frontmatter that defines a reusable workflow. Unlike Tools (TypeScript code), Skills are *declarative*: prompt + permission config.

**Key fields:**
- `name` — unique slug, defaults to directory name
- `description` — shown in discovery listing
- `paths` — gitignore patterns for conditional activation
- `disableModelInvocation` — hidden from AI but user can still `/name`
- `allowedTools` — temporarily whitelisted during execution

**Source:** `src/types/types.ts`

In [ ]:
@dataclass
class SkillFrontmatter:
    name: Optional[str] = None
    description: Optional[str] = None
    when_to_use: Optional[str] = None
    allowedTools: list = field(default_factory=list)
    argumentHint: Optional[str] = None
    disableModelInvocation: bool = False
    paths: Optional[list] = None
    hasForkContext: bool = False
    raw: dict = field(default_factory=dict)


@dataclass
class Skill:
    name: str = ""
    description: str = ""
    whenToUse: Optional[str] = None
    body: str = ""
    filePath: str = ""
    baseDir: str = ""
    source: str = ""  # "user" or "project"
    frontmatter: SkillFrontmatter = field(default_factory=SkillFrontmatter)


print("Skill types defined: SkillFrontmatter, Skill")

## 15. Skills — Frontmatter Parsing

The parser splits `---\n...\n---\n<body>` into YAML frontmatter and markdown body. Never throws — invalid YAML is reported via `parseError`.

**Normalization behaviors:**
- `allowed-tools` accepts YAML arrays and CSV strings (`"Read, Grep, Glob"`)
- `paths` is always normalized to an array
- Boolean fields accept `true`/`yes`/`1`
- Unknown fields preserved in `raw` for forward compatibility

**Source:** `src/services/skills/parseFrontmatter.ts`

In [ ]:
import yaml

FRONTMATTER_RE = re.compile(r"^---\r?\n([\s\S]*?)\r?\n---\r?\n?([\s\S]*)$")


def split_frontmatter(content: str) -> dict:
    """Split a SKILL.md into frontmatter + body. Never throws."""
    match = FRONTMATTER_RE.match(content)
    if not match:
        return {"raw": {}, "body": content, "parseError": None}

    yaml_text, body = match.group(1), match.group(2)
    try:
        parsed = yaml.safe_load(yaml_text)
        if parsed and isinstance(parsed, dict):
            return {"raw": parsed, "body": body, "parseError": None}
        return {"raw": {}, "body": body, "parseError": "Frontmatter must be a YAML mapping"}
    except Exception as e:
        return {"raw": {}, "body": body, "parseError": str(e)}


def as_string(value) -> Optional[str]:
    if isinstance(value, str):
        trimmed = value.strip()
        return trimmed if trimmed else None
    if isinstance(value, (int, float, bool)):
        return str(value)
    return None


def as_string_array(value) -> list:
    if isinstance(value, list):
        return [item.strip() for item in value if isinstance(item, str) and item.strip()]
    if isinstance(value, str):
        return [s.strip() for s in value.split(",") if s.strip()]
    return []


def as_boolean(value) -> bool:
    if isinstance(value, bool):
        return value
    if isinstance(value, str):
        return value.strip().lower() in ("true", "yes", "1")
    return False


def extract_fallback_description(body: str) -> str:
    """Extract the first non-empty paragraph (skipping headings)."""
    lines = body.split("\n")
    buf = []
    for raw_line in lines:
        line = raw_line.strip()
        if not line:
            if buf:
                break
            continue
        if not buf and line.startswith("#"):
            continue
        buf.append(line)
    return " ".join(buf).strip()


def normalize_frontmatter(raw: dict, body: str) -> SkillFrontmatter:
    """Convert raw YAML map into a typed SkillFrontmatter."""
    allowed_tools = as_string_array(raw.get("allowed-tools") or raw.get("allowedTools"))
    paths = as_string_array(raw.get("paths"))

    return SkillFrontmatter(
        name=as_string(raw.get("name")),
        description=as_string(raw.get("description")),
        when_to_use=as_string(raw.get("when_to_use") or raw.get("whenToUse")),
        allowedTools=allowed_tools,
        argumentHint=as_string(raw.get("argument-hint") or raw.get("argumentHint")),
        disableModelInvocation=as_boolean(raw.get("disable-model-invocation") or raw.get("disableModelInvocation")),
        paths=paths if paths else None,
        hasForkContext=as_string(raw.get("context")) == "fork",
        raw=raw,
    )


# Demo
sample_skill_md = '''---
name: code-review
description: Analyzes code for bugs, style issues, and security vulnerabilities
when_to_use: When reviewing PRs or code snippets
allowed-tools: Read, Grep, Glob
paths:
  - "**/*.ts"
  - "**/*.py"
---

# Code Review Skill

This skill performs a thorough code review...
'''

split = split_frontmatter(sample_skill_md)
print(f"Parse error: {split['parseError']}")
print(f"Raw frontmatter keys: {list(split['raw'].keys())}")
print(f"Body preview: {split['body'][:60]}...")

fm = normalize_frontmatter(split["raw"], split["body"])
print(f"\nNormalized frontmatter:")
print(f"  name: {fm.name}")
print(f"  description: {fm.description}")
print(f"  when_to_use: {fm.when_to_use}")
print(f"  allowedTools: {fm.allowedTools}")
print(f"  paths: {fm.paths}")
print(f"  disableModelInvocation: {fm.disableModelInvocation}")

## 16. Skills Registry — Dual-Map Architecture

The registry maintains two maps:
- **`dynamic`** — skills visible to the model (in system prompt listing)
- **`conditional`** — skills with `paths` that haven't matched yet (hidden from model)

Activation is **one-way and sticky**: once a conditional skill is promoted to dynamic, it stays visible for the process lifetime.

**Source:** `src/services/skills/registry.ts`

In [ ]:
class SkillsRegistry:
    """Central in-memory state for loaded skills, split into dynamic/conditional maps."""

    def __init__(self):
        self._dynamic: dict[str, Skill] = {}
        self._conditional: dict[str, Skill] = {}
        self._initialized = False

    def set_skills(self, skills: list[Skill]) -> None:
        """Replace the registry with a freshly loaded set. Called once at startup."""
        self._dynamic.clear()
        self._conditional.clear()
        for skill in skills:
            if skill.frontmatter.paths and len(skill.frontmatter.paths) > 0:
                self._conditional[skill.name] = skill
            else:
                self._dynamic[skill.name] = skill
        self._initialized = True

    def get_model_visible_skills(self) -> list[Skill]:
        """Skills visible to the model (excludes disableModelInvocation)."""
        return [s for s in self._dynamic.values() if not s.frontmatter.disableModelInvocation]

    def get_all_user_invocable_skills(self) -> list[Skill]:
        """All skills the user can invoke via /<name>."""
        return list(self._dynamic.values()) + list(self._conditional.values())

    def find_skill(self, name: str) -> Optional[Skill]:
        """Look up by name across both maps."""
        return self._dynamic.get(name) or self._conditional.get(name)

    def activate_conditional(self, name: str) -> bool:
        """Promote a conditional skill to dynamic. Returns True if newly activated."""
        skill = self._conditional.get(name)
        if not skill:
            return False
        del self._conditional[name]
        self._dynamic[name] = skill
        return True

    def list_conditional_skills(self) -> list[Skill]:
        return list(self._conditional.values())

    def __repr__(self):
        return f"SkillsRegistry(dynamic={len(self._dynamic)}, conditional={len(self._conditional)})"


# Demo
skills_registry = SkillsRegistry()

demo_skills = [
    Skill(name="code-review", description="Review code", source="user",
          frontmatter=SkillFrontmatter()),
    Skill(name="refactor", description="Refactor code", source="user",
          frontmatter=SkillFrontmatter(disableModelInvocation=True)),
    Skill(name="test-reviewer", description="Review tests", source="project",
          frontmatter=SkillFrontmatter(paths=["**/*.test.ts", "**/*.spec.ts"])),
    Skill(name="deploy-helper", description="Deploy assistance", source="project",
          frontmatter=SkillFrontmatter(paths=["**/deploy/**"])),
]

skills_registry.set_skills(demo_skills)
print(f"Registry: {skills_registry}")
print(f"Model-visible: {[s.name for s in skills_registry.get_model_visible_skills()]}")
print(f"User-invocable: {[s.name for s in skills_registry.get_all_user_invocable_skills()]}")

# Activate a conditional skill
activated = skills_registry.activate_conditional("test-reviewer")
print(f"\nActivated 'test-reviewer': {activated}")
print(f"After activation: {skills_registry}")
print(f"Model-visible: {[s.name for s in skills_registry.get_model_visible_skills()]}")

## 17. Conditional Skill Activation — Path-Based Triggers

When the agent touches a file (via Read/Write/Edit/Glob), the system checks if any conditional skill's `paths` patterns match. Uses gitignore-style matching via the `pathspec` library.

**Source:** `src/services/skills/conditional.ts`

In [ ]:
import pathspec


def activate_conditional_skills_for_paths(
    file_paths: list[str],
    cwd: str,
    registry: SkillsRegistry,
) -> list[str]:
    """Activate conditional skills whose path patterns match touched files."""
    if not file_paths:
        return []
    candidates = registry.list_conditional_skills()
    if not candidates:
        return []

    # Convert to repo-relative paths for gitignore matching
    cwd_path = Path(cwd)
    relative_paths = []
    for p in file_paths:
        abs_path = Path(p) if Path(p).is_absolute() else (cwd_path / p)
        try:
            rel = abs_path.relative_to(cwd_path)
            relative_paths.append(str(rel).replace("\\", "/"))
        except ValueError:
            continue

    if not relative_paths:
        return []

    activated = []
    for skill in candidates:
        patterns = skill.frontmatter.paths
        if not patterns:
            continue
        spec = pathspec.PathSpec.from_lines("gitwildmatch", patterns)
        if any(spec.match_file(p) for p in relative_paths):
            if registry.activate_conditional(skill.name):
                activated.append(skill.name)

    return activated


def extract_tool_file_paths(tool_name: str, tool_input: dict) -> list[str]:
    """Extract file paths from well-known tool inputs."""
    paths = []
    if tool_name in ("Read", "Write", "Edit"):
        fp = tool_input.get("file_path")
        if isinstance(fp, str):
            paths.append(fp)
    elif tool_name == "Glob":
        root = tool_input.get("path")
        if isinstance(root, str):
            paths.append(root)
    return paths


# Demo: Reset registry and test activation
skills_registry_2 = SkillsRegistry()
skills_registry_2.set_skills([
    Skill(name="py-linter", description="Lint Python", source="project",
          frontmatter=SkillFrontmatter(paths=["**/*.py"])),
    Skill(name="ts-helper", description="TypeScript help", source="project",
          frontmatter=SkillFrontmatter(paths=["src/**/*.ts", "src/**/*.tsx"])),
    Skill(name="always-on", description="General skill", source="user",
          frontmatter=SkillFrontmatter()),
])

print(f"Before: {skills_registry_2}")

# Simulate touching a Python file
touched = ["/home/user/project/src/utils/helper.py"]
activated = activate_conditional_skills_for_paths(touched, "/home/user/project", skills_registry_2)
print(f"Touched: {touched}")
print(f"Activated: {activated}")
print(f"After: {skills_registry_2}")

# Simulate touching a TypeScript file
touched2 = ["/home/user/project/src/components/App.tsx"]
activated2 = activate_conditional_skills_for_paths(touched2, "/home/user/project", skills_registry_2)
print(f"\nTouched: {touched2}")
print(f"Activated: {activated2}")
print(f"After: {skills_registry_2}")

## 18. Skills Budget — System Prompt Injection

The budget module formats skill descriptions into the system prompt with a **three-tier degradation** strategy:

| Tier | Format | When |
|------|--------|------|
| 1 (full) | `- name: description (≤250 chars)` | Total fits budget |
| 2 (shrunk) | `- name: description (shared budget)` | Shrink equally, ≥20 chars each |
| 3 (names) | `- name` | Last resort |

Default budget: 8,000 characters (~2,000 tokens for a 200K model).

**Source:** `src/services/skills/budget.ts`

In [ ]:
MAX_LISTING_DESC_CHARS = 250
MIN_DESC_CHARS_PER_SKILL = 20
DEFAULT_BUDGET_CHARS = 8000


def get_skill_char_budget() -> int:
    env_value = os.environ.get("EASY_AGENT_SKILL_CHAR_BUDGET")
    if env_value:
        try:
            parsed = int(env_value)
            if parsed > 0:
                return parsed
        except ValueError:
            pass
    return DEFAULT_BUDGET_CHARS


def truncate_desc(desc: str, max_len: int) -> str:
    if len(desc) <= max_len:
        return desc
    if max_len <= 1:
        return "..."
    return desc[:max_len - 1].rstrip() + "..."


def build_skill_line(skill: Skill, desc_max: int) -> str:
    capped_max = min(desc_max, MAX_LISTING_DESC_CHARS)
    full_desc = f"{skill.description} — {skill.whenToUse}" if skill.whenToUse else skill.description
    desc = truncate_desc(full_desc, capped_max)
    return f"- {skill.name}: {desc}"


def format_skills_within_budget(skills: list[Skill], budget: int = None) -> str:
    """Render discovery listing under the given char budget (3-tier degradation)."""
    if not skills:
        return ""
    budget = budget or get_skill_char_budget()

    # Tier 1: full descriptions
    tier1 = [build_skill_line(s, MAX_LISTING_DESC_CHARS) for s in skills]
    tier1_total = sum(len(line) + 1 for line in tier1)
    if tier1_total <= budget:
        return "\n".join(tier1)

    # Tier 2: shrink descriptions equally
    prefix_cost = sum(len(f"- {s.name}: ") + 1 for s in skills)
    desc_budget = budget - prefix_cost
    if desc_budget >= len(skills) * MIN_DESC_CHARS_PER_SKILL:
        per_desc = max(MIN_DESC_CHARS_PER_SKILL, desc_budget // len(skills))
        tier2 = [build_skill_line(s, per_desc) for s in skills]
        tier2_total = sum(len(line) + 1 for line in tier2)
        if tier2_total <= budget:
            return "\n".join(tier2)

    # Tier 3: names only
    return "\n".join(f"- {s.name}" for s in skills)


def format_skills_system_reminder(skills: list[Skill]) -> str:
    """Build the system-reminder block for injection into every system prompt."""
    if not skills:
        return ""
    listing = format_skills_within_budget(skills)
    if not listing:
        return ""
    return "\n".join([
        "<system-reminder>",
        "Available skills you can invoke via the `Skill` tool.",
        'Call `Skill(skill="<name>", args="<optional args>")` when the user\'s request matches.',
        "",
        listing,
        "</system-reminder>",
    ])


# Demo: show all three tiers
demo_skills_for_budget = [
    Skill(name="code-review", description="Analyzes code for bugs, style issues, and security vulnerabilities",
          whenToUse="When reviewing PRs", frontmatter=SkillFrontmatter()),
    Skill(name="refactor-helper", description="Suggests and applies refactoring patterns to improve code quality",
          frontmatter=SkillFrontmatter()),
    Skill(name="test-writer", description="Generates comprehensive test suites for existing code",
          whenToUse="When adding tests", frontmatter=SkillFrontmatter()),
    Skill(name="doc-generator", description="Creates documentation from source code and comments",
          frontmatter=SkillFrontmatter()),
]

print("=== Tier 1 (large budget) ===")
print(format_skills_within_budget(demo_skills_for_budget, budget=2000))
print(f"\n=== Tier 2 (tight budget, 400 chars) ===")
print(format_skills_within_budget(demo_skills_for_budget, budget=400))
print(f"\n=== Tier 3 (very tight budget, 100 chars) ===")
print(format_skills_within_budget(demo_skills_for_budget, budget=100))

print(f"\n=== Full system reminder ===")
print(format_skills_system_reminder(demo_skills_for_budget[:2]))

## 19. Stream Debug Infrastructure

Opt-in logging of every raw SSE event for debugging provider compatibility. Activated by `EASY_AGENT_DEBUG_STREAM=1`. Outputs JSONL to `~/.easy-agent/stream-debug.log`.

**Key invariant:** Logging must never throw or affect the stream itself.

| Record Kind | When | Payload |
|-------------|------|---------|
| `request` | Before streaming starts | model, messageCount, toolNames |
| `event` | Each raw SSE event | Full event object |
| `assembled` | After stream completes | stopReason, blockCount |
| `stream_error` | On catch | error message |

**Source:** `src/utils/streamDebug.ts`

In [ ]:
from datetime import datetime


class StreamDebugLogger:
    """Stream debug logger — no-op when disabled, JSONL when enabled."""

    def __init__(self, enabled: bool = False, log_path: Optional[str] = None):
        self.enabled = enabled
        self.log_path = log_path
        self._records: list[dict] = []  # In-memory for demo

    def write(self, kind: str, payload: Any) -> None:
        """Append a single JSON record. Safe to call when disabled."""
        if not self.enabled:
            return
        try:
            record = {"ts": datetime.now().isoformat(), "kind": kind, "payload": payload}
            self._records.append(record)
            # In real code: appendFileSync(log_path, json.dumps(record) + "\n")
        except Exception:
            pass  # Logging must never break the stream

    def get_records(self) -> list[dict]:
        return self._records


# Demo
logger = StreamDebugLogger(enabled=True)
logger.write("request", {"model": "claude-sonnet-4-20250514", "messageCount": 3, "toolNames": ["Read", "Write"]})
logger.write("event", {"type": "message_start", "message": {"id": "msg_01234"}})
logger.write("event", {"type": "content_block_delta", "delta": {"type": "text_delta", "text": "Hello"}})
logger.write("assembled", {"stopReason": "end_turn", "blockCount": 1})

print(f"Debug records captured: {len(logger.get_records())}")
for r in logger.get_records():
    print(f"  [{r['kind']}] {json.dumps(r['payload'])[:80]}")

## 20. Integration — How the Layer Connects to the Agentic Loop

The agentic loop (`src/core/agenticLoop.ts`) calls into the communication layer at these points:

```
┌─────────────────────────────────────────────────────────────────┐
│  Core Agentic Loop (Layer 3)                                    │
│                                                                 │
│  query() generator                                              │
│    │                                                            │
│    ├── streamMessage(messages, tools, system)  ←── API Layer    │
│    │     yields StreamEvents → UI renders incrementally         │
│    │     returns StreamResult → assemble assistant message      │
│    │                                                            │
│    ├── tool_use detected?                                       │
│    │     ├── isMcpToolName(name)?  ─── route to MCP adapter    │
│    │     └── local tool?           ─── execute directly         │
│    │                                                            │
│    ├── activateConditionalSkillsForPaths(touched_files)         │
│    │     (after each tool execution)                            │
│    │                                                            │
│    └── token budget check → auto-compact if over threshold      │
│                                                                 │
│  Skills visible in system prompt via formatSkillsSystemReminder │
└─────────────────────────────────────────────────────────────────┘
```

**Key invariants:**
1. `streamMessage()` is the ONLY path to the LLM — both streaming and retry go through it
2. MCP tools are indistinguishable from local tools after adapter wrapping
3. Conditional skills activate lazily — no upfront cost for large skill sets
4. Token budget checks happen EVERY turn (except the first) to prevent context overflow

## Summary — Source File Mapping

| Notebook Section | Source File(s) | Key Function/Class |
|-----------------|----------------|-------------------|
| §2 Message Types | `src/types/message.ts` | ContentBlock, StreamEvent, Usage |
| §3 Stream Events | `src/types/message.ts` | StreamEvent union (6 variants) |
| §4 API Client | `src/services/api/client.ts` | `getAnthropicClient()`, singleton pattern |
| §5 Streaming Engine | `src/services/api/streaming.ts` | `streamMessage()` — the core primitive |
| §5.2 Non-streaming | `src/services/api/streaming.ts` | `createMessage()`, `streamMessageWithRetry()` |
| §6 Token Estimation | `src/utils/tokens.ts` | `estimateMessageTokens()`, heuristic ratios |
| §6.1 Budget Snapshot | `src/utils/tokens.ts` | `buildTokenBudgetSnapshot()` |
| §7 MCP Types | `src/types/mcp.ts` | Config + connection state types |
| §8 Name Normalization | `src/services/mcp/normalization.ts`, `mcpStringUtils.ts` | `normalizeNameForMCP()`, `buildMcpToolName()` |
| §9 Config Loading | `src/services/mcp/config.ts` | `loadMcpConfigs()`, `validateServerConfig()` |
| §10 MCP Registry | `src/services/mcp/registry.ts` | `setMcpRegistryEntry()`, `getMcpRegistry()` |
| §11 MCP Bootstrap | `src/services/mcp/bootstrap.ts` | `bootstrapMcp()`, parallel connection |
| §12 Tool Adapter | `src/services/mcp/fetchTools.ts` | `buildToolAdapter()`, `fetchToolsForConnection()` |
| §13 Client Connection | `src/services/mcp/client.ts` | `connectToServer()`, `escalatedKill()` |
| §14 Skill Types | `src/types/types.ts` | `Skill`, `SkillFrontmatter` |
| §15 Frontmatter | `src/services/skills/parseFrontmatter.ts` | `splitFrontmatter()`, `normalizeFrontmatter()` |
| §16 Skills Registry | `src/services/skills/registry.ts` | dual-map, `activateConditional()` |
| §17 Conditional | `src/services/skills/conditional.ts` | `activateConditionalSkillsForPaths()` |
| §18 Budget Format | `src/services/skills/budget.ts` | `formatSkillsWithinBudget()` (3-tier) |
| §19 Stream Debug | `src/utils/streamDebug.ts` | `writeStreamDebug()` — opt-in JSONL |

**Simplifications made:**
- AbortSignal support removed from streaming
- Process cleanup (SIGINT/SIGTERM registration) shown conceptually, not wired
- MCP SDK connections simulated (would require actual MCP servers)
- Stream debug writes to in-memory list instead of filesystem
- Retry/backoff logic in `streamMessageWithRetry` shown at API level only